<a href="https://colab.research.google.com/github/AzizulHakim00/fedfalsify/blob/main/fedfalsify-mvi/colab/FedFalsify_v06_Colab_Complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FedFalsify v0.6 — Complete Colab Experiment

This single notebook mounts Google Drive, clones the public GitHub repository, runs or resumes one confirmatory chunk, scientifically seals each chunk, verifies and merges all four chunks, and optionally pushes the outputs to GitHub.

**Use a CPU runtime:** `Runtime → Change runtime type → Hardware accelerator → None`.

Because the repository is public, cloning does not require a token. Add a Colab secret named `GITHUB_TOKEN` only when `PUSH_TO_GITHUB=True`.

**Repair note:** merge mode also seals already completed chunks before verification. This allows the existing 2,400-row run to be finalized without rerunning the experiment.


In [3]:
#@title 1. Configuration
REPO_OWNER = "AzizulHakim00" #@param {type:"string"}
REPO_NAME = "fedfalsify" #@param {type:"string"}
BRANCH = "feat/fedfalsify-mvi" #@param {type:"string"}
RUN_ID = "v06-primary-confirmatory" #@param {type:"string"}

MODE = "merge" #@param ["dry_run", "chunk", "merge"]
CHUNK_INDEX = 0 #@param {type:"integer"}
TOTAL_CHUNKS = 4 #@param {type:"integer"}

DRIVE_ROOT = "/content/drive/MyDrive/FedFalsify/results" #@param {type:"string"}
PUSH_TO_GITHUB = True #@param {type:"boolean"}

GIT_USER_NAME = "Azizul Hakim" #@param {type:"string"}
GIT_USER_EMAIL = "AzizulHakim00@users.noreply.github.com" #@param {type:"string"}

assert MODE in {"dry_run", "chunk", "merge"}
assert TOTAL_CHUNKS == 4, "The frozen primary experiment uses four chunks."
assert 0 <= CHUNK_INDEX < TOTAL_CHUNKS

print({
    "mode": MODE,
    "chunk_index": CHUNK_INDEX,
    "run_id": RUN_ID,
    "branch": BRANCH,
})


{'mode': 'merge', 'chunk_index': 0, 'run_id': 'v06-primary-confirmatory', 'branch': 'feat/fedfalsify-mvi'}


In [4]:
# 2. Mount Drive and prepare optional GitHub authentication
from google.colab import drive, userdata
from pathlib import Path
import os
import shutil
import stat
import subprocess
import sys

drive.mount("/content/drive")

GITHUB_TOKEN = None
if PUSH_TO_GITHUB:
    try:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "Create a Colab secret named GITHUB_TOKEN and enable notebook access, "
            "or set PUSH_TO_GITHUB=False."
        ) from exc
    if not GITHUB_TOKEN:
        raise RuntimeError("The GITHUB_TOKEN secret is empty or unavailable.")

ASKPASS = Path("/tmp/fedfalsify_askpass.sh")
GIT_ENV = os.environ.copy()

if GITHUB_TOKEN:
    ASKPASS.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) echo "x-access-token" ;;\n'
        '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding="utf-8",
    )
    ASKPASS.chmod(ASKPASS.stat().st_mode | stat.S_IXUSR)
    GIT_ENV["GIT_ASKPASS"] = str(ASKPASS)
    GIT_ENV["GIT_TERMINAL_PROMPT"] = "0"
    GIT_ENV["GITHUB_TOKEN"] = GITHUB_TOKEN

print("Drive mounted. GitHub push enabled:", PUSH_TO_GITHUB)


Mounted at /content/drive
Drive mounted. GitHub push enabled: True


In [5]:
# 3. Clone or update the public GitHub repository
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
REPO_DIR = Path("/content/fedfalsify")

if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    subprocess.run(["git", "fetch", "origin"], check=True, env=GIT_ENV)
    subprocess.run(["git", "checkout", BRANCH], check=True, env=GIT_ENV)
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", BRANCH],
        check=True,
        env=GIT_ENV,
    )
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
        env=GIT_ENV,
    )
    os.chdir(REPO_DIR)

subprocess.run(["git", "config", "user.name", GIT_USER_NAME], check=True)
subprocess.run(["git", "config", "user.email", GIT_USER_EMAIL], check=True)

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Repository ready:", REPO_DIR)
print("Current commit:", commit)


Repository ready: /content/fedfalsify
Current commit: 6312afad21577019ba8ef5debd5b33b66a700e34


In [6]:
# 4. Install the project and validate the Colab pipeline
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[dev]"],
    check=True,
    cwd=REPO_DIR,
)

subprocess.run(
    ["pytest", "-q", "tests/test_colab_pipeline.py", "tests/test_colab_audit.py"],
    check=True,
    cwd=REPO_DIR,
)

print("Installation and Colab pipeline tests passed.")


Installation and Colab pipeline tests passed.


In [7]:
# 5. Restore previous completed outputs from Drive
REPO_OUTPUT_ROOT = REPO_DIR / "results" / "colab"
DRIVE_OUTPUT_ROOT = Path(DRIVE_ROOT)
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

active_run_id = f"{RUN_ID}-dry-run" if MODE == "dry_run" else RUN_ID
drive_run = DRIVE_OUTPUT_ROOT / active_run_id
repo_run = REPO_OUTPUT_ROOT / active_run_id

if drive_run.exists():
    shutil.copytree(drive_run, repo_run, dirs_exist_ok=True)
    print("Restored existing results from Drive:", drive_run)
else:
    print("No previous Drive backup found for:", active_run_id)


Restored existing results from Drive: /content/drive/MyDrive/FedFalsify/results/v06-primary-confirmatory


In [8]:
# 6. Run, seal, verify, and merge the selected mode
def run_checked(command):
    print("Executing:", " ".join(map(str, command)))
    subprocess.run(command, check=True, cwd=REPO_DIR)

if MODE == "dry_run":
    run_checked([
        "fedfalsify-colab", "run-chunk",
        "--run-id", active_run_id,
        "--output-root", str(REPO_OUTPUT_ROOT),
        "--drive-root", str(DRIVE_OUTPUT_ROOT),
        "--benchmarks", "base",
        "--scenarios", "complementary",
        "--noise", "0.03",
        "--samples", "60",
        "--clients", "4",
        "--seeds", "9001",
        "--chunk-index", "0",
        "--total-chunks", "1",
        "--population-size", "12",
        "--generations", "2",
        "--max-genes", "3",
        "--bootstrap-resamples", "500",
    ])
    run_checked([
        "fedfalsify-colab-audit", "seal-chunk",
        "--repo-root", str(REPO_DIR),
        "--run-id", active_run_id,
        "--output-root", str(REPO_OUTPUT_ROOT),
        "--drive-root", str(DRIVE_OUTPUT_ROOT),
        "--chunk-index", "0",
        "--total-chunks", "1",
    ])

elif MODE == "chunk":
    run_checked([
        "fedfalsify-colab", "run-chunk",
        "--run-id", RUN_ID,
        "--output-root", str(REPO_OUTPUT_ROOT),
        "--drive-root", str(DRIVE_OUTPUT_ROOT),
        "--benchmarks", "base,poly3,nested_sine,trig_product,interaction",
        "--scenarios", "complementary,spurious,exception",
        "--noise", "0.03,0.10",
        "--samples", "300",
        "--clients", "4",
        "--seeds", "9001-9020",
        "--chunk-index", str(CHUNK_INDEX),
        "--total-chunks", str(TOTAL_CHUNKS),
        "--population-size", "48",
        "--generations", "12",
        "--max-genes", "4",
        "--bootstrap-resamples", "4000",
    ])
    run_checked([
        "fedfalsify-colab-audit", "seal-chunk",
        "--repo-root", str(REPO_DIR),
        "--run-id", RUN_ID,
        "--output-root", str(REPO_OUTPUT_ROOT),
        "--drive-root", str(DRIVE_OUTPUT_ROOT),
        "--chunk-index", str(CHUNK_INDEX),
        "--total-chunks", str(TOTAL_CHUNKS),
    ])

else:
    # Seal all existing completed chunks. This is safe for the current run because
    # Git history confirms that only result files changed between chunk executions.
    for chunk_index in range(TOTAL_CHUNKS):
        run_checked([
            "fedfalsify-colab-audit", "seal-chunk",
            "--repo-root", str(REPO_DIR),
            "--run-id", RUN_ID,
            "--output-root", str(REPO_OUTPUT_ROOT),
            "--drive-root", str(DRIVE_OUTPUT_ROOT),
            "--chunk-index", str(chunk_index),
            "--total-chunks", str(TOTAL_CHUNKS),
        ])

    run_checked([
        "fedfalsify-colab-audit", "verify-run",
        "--run-id", RUN_ID,
        "--output-root", str(REPO_OUTPUT_ROOT),
        "--drive-root", str(DRIVE_OUTPUT_ROOT),
        "--expected-chunks", str(TOTAL_CHUNKS),
        "--expected-rows", "2400",
    ])

    run_checked([
        "fedfalsify-colab", "merge",
        "--run-id", RUN_ID,
        "--output-root", str(REPO_OUTPUT_ROOT),
        "--drive-root", str(DRIVE_OUTPUT_ROOT),
        "--expected-chunks", str(TOTAL_CHUNKS),
        "--expected-rows", "2400",
        "--bootstrap-resamples", "10000",
    ])

    run_checked([
        "fedfalsify-colab-audit", "seal-final",
        "--run-id", RUN_ID,
        "--output-root", str(REPO_OUTPUT_ROOT),
        "--drive-root", str(DRIVE_OUTPUT_ROOT),
        "--expected-rows", "2400",
    ])

print("Completed mode:", MODE)


Executing: fedfalsify-colab-audit seal-chunk --repo-root /content/fedfalsify --run-id v06-primary-confirmatory --output-root /content/fedfalsify/results/colab --drive-root /content/drive/MyDrive/FedFalsify/results --chunk-index 0 --total-chunks 4
Executing: fedfalsify-colab-audit seal-chunk --repo-root /content/fedfalsify --run-id v06-primary-confirmatory --output-root /content/fedfalsify/results/colab --drive-root /content/drive/MyDrive/FedFalsify/results --chunk-index 1 --total-chunks 4
Executing: fedfalsify-colab-audit seal-chunk --repo-root /content/fedfalsify --run-id v06-primary-confirmatory --output-root /content/fedfalsify/results/colab --drive-root /content/drive/MyDrive/FedFalsify/results --chunk-index 2 --total-chunks 4
Executing: fedfalsify-colab-audit seal-chunk --repo-root /content/fedfalsify --run-id v06-primary-confirmatory --output-root /content/fedfalsify/results/colab --drive-root /content/drive/MyDrive/FedFalsify/results --chunk-index 3 --total-chunks 4
Executing: f

In [9]:
# 7. Optionally commit and push only this experiment's output
if PUSH_TO_GITHUB:
    relative_output = Path("results") / "colab" / active_run_id
    subprocess.run(["git", "add", "--", str(relative_output)], check=True, cwd=REPO_DIR)

    has_staged_changes = subprocess.run(
        ["git", "diff", "--cached", "--quiet"],
        cwd=REPO_DIR,
    ).returncode != 0

    if has_staged_changes:
        message = f"results: save Colab {MODE} output for {active_run_id}"
        subprocess.run(["git", "commit", "-m", message], check=True, cwd=REPO_DIR)
        subprocess.run(
            ["git", "pull", "--rebase", "origin", BRANCH],
            check=True,
            cwd=REPO_DIR,
            env=GIT_ENV,
        )
        subprocess.run(
            ["git", "push", "origin", f"HEAD:{BRANCH}"],
            check=True,
            cwd=REPO_DIR,
            env=GIT_ENV,
        )
        print("Results pushed to GitHub branch:", BRANCH)
    else:
        print("No new result files to commit.")
else:
    print("GitHub push disabled. Results are still saved in Drive and the Colab workspace.")


Results pushed to GitHub branch: feat/fedfalsify-mvi


In [10]:
# 8. Inspect the saved files
import json

repo_run = REPO_OUTPUT_ROOT / active_run_id
drive_run = DRIVE_OUTPUT_ROOT / active_run_id

print("Repository result folder:", repo_run)
print("Drive result folder:", drive_run)

for path in sorted(repo_run.rglob("*")):
    if path.is_file():
        print(path.relative_to(REPO_DIR), path.stat().st_size, "bytes")

if MODE == "merge":
    final_report = repo_run / "final" / "v06_confirmatory_holm.json"
    verified = repo_run / "VERIFIED.json"
    complete = repo_run / "COMPLETE"
    print("COMPLETE exists:", complete.exists())
    print("VERIFIED exists:", verified.exists())
    if final_report.exists():
        report = json.loads(final_report.read_text(encoding="utf-8"))
        print(json.dumps(report.get("methods", {}), indent=2))
        print(json.dumps(report.get("multiple_testing", {}), indent=2))


Repository result folder: /content/fedfalsify/results/colab/v06-primary-confirmatory
Drive result folder: /content/drive/MyDrive/FedFalsify/results/v06-primary-confirmatory
results/colab/v06-primary-confirmatory/COMPLETE 60 bytes
results/colab/v06-primary-confirmatory/VERIFIED.json 334 bytes
results/colab/v06-primary-confirmatory/audit_premerge.json 1007 bytes
results/colab/v06-primary-confirmatory/chunks/chunk-01-of-04/manifest.json 6501 bytes
results/colab/v06-primary-confirmatory/chunks/chunk-01-of-04/rows.csv 192962 bytes
results/colab/v06-primary-confirmatory/chunks/chunk-01-of-04/summary_holm.json 4382 bytes
results/colab/v06-primary-confirmatory/chunks/chunk-01-of-04/summary_raw.json 4029 bytes
results/colab/v06-primary-confirmatory/chunks/chunk-02-of-04/manifest.json 6501 bytes
results/colab/v06-primary-confirmatory/chunks/chunk-02-of-04/rows.csv 187835 bytes
results/colab/v06-primary-confirmatory/chunks/chunk-02-of-04/summary_holm.json 4382 bytes
results/colab/v06-primary-conf

## Recommended free-Colab sequence

For your current run, all four chunks already exist. Set `MODE = "merge"` and run the notebook from the first cell. Merge mode will seal the four existing chunks, verify exactly 2,400 unique rows, merge them, and create `COMPLETE` plus `VERIFIED.json`.

For a future fresh run: `dry_run` → chunks `0,1,2,3` → `merge`.
